In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/malicious_phish.csv')

# Display the first few rows of the dataframe
df.head()

In [ ]:
# Get the shape of the dataframe
print(f"Shape of the dataframe: {df.shape}")

# Get information about the columns
print("\nInformation about the dataframe:")
df.info()

# Get the value counts of the 'type' column
print("\nDistribution of URL types:")
df['type'].value_counts()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
from urllib.parse import urlparse
import re

# Function to extract features
def extract_features(url):
    features = {}
    
    # Parse the URL
    parsed_url = urlparse(url)
    
    # Extract features
    features['url_length'] = len(url)
    features['hostname_length'] = len(parsed_url.hostname) if parsed_url.hostname else 0
    features['path_length'] = len(parsed_url.path)
    
    # First directory length
    path_parts = parsed_url.path.split('/')
    features['fd_length'] = len(path_parts[1]) if len(path_parts) > 1 else 0
    
    # TLD length
    if parsed_url.hostname:
        tld = parsed_url.hostname.split('.')[-1]
        features['tld_length'] = len(tld)
    else:
        features['tld_length'] = 0
    
    features['count_dash'] = url.count('-')
    features['count_at'] = url.count('@')
    features['count_question'] = url.count('?')
    features['count_percent'] = url.count('%')
    features['count_dot'] = url.count('.')
    features['count_equal'] = url.count('=')
    features['count_http'] = url.count('http')
    features['count_https'] = url.count('https')
    features['count_www'] = url.count('www')
    features['count_digits'] = len(re.findall(r'\d', url))
    features['count_letters'] = len(re.findall(r'[a-zA-Z]', url))
    features['count_dir'] = parsed_url.path.count('/')
    
    return features

# Apply the function to the url column
features_df = df['url'].apply(lambda x: pd.Series(extract_features(x)))

# Concatenate the features dataframe with the original dataframe
df = pd.concat([df, features_df], axis=1)

# Display the first few rows with the new features
df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode the 'type' column
le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])

# Define features (X) and target (y)
X = df.drop(['url', 'type', 'type_encoded'], axis=1)
y = df['type_encoded']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the shapes of the training and testing sets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create an instance of the RandomForestClassifier
rfc = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rfc.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on the test set
y_pred = rfc.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Print the classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))